# Módulo 02 · Datos no estructurados: clustering y reducción de dimensión

**Casos:**
- **Segmentación de clientes** de un e-commerce a partir de su comportamiento de compra (RFM: *recency, frequency, monetary*).
- **Visualizar datos de alta dimensión**: imágenes de dígitos de 64 píxeles proyectadas a 2D.

Objetivos:
1. Entender por qué hay que **escalar** antes de medir distancias.
2. Aplicar k-means y elegir *k* con el método del codo y el coeficiente de silueta.
3. Usar clustering jerárquico y leer un dendrograma.
4. Reducir dimensión con PCA y t-SNE, e interpretar sus resultados.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.datasets import load_digits
from scipy.cluster.hierarchy import linkage, dendrogram

sns.set_theme(style="whitegrid")
rng = np.random.default_rng(0)

## 1. Datos de clientes (RFM)
Simulamos 1.500 clientes con cuatro perfiles "ocultos" que el algoritmo tendrá que descubrir sin etiquetas.

In [ ]:
perfiles = {
    #               recencia(días) frecuencia(compras/año) gasto_medio(€)
    "fieles":       (15,  25, 60),
    "ocasionales":  (90,   5, 45),
    "premium":      (30,  10, 250),
    "dormidos":     (300,  2, 35),
}
filas = []
for nombre, (r, f, m) in perfiles.items():
    k = 375
    filas.append(pd.DataFrame({
        "recencia": rng.gamma(4, r / 4, k),
        "frecuencia": rng.poisson(f, k) + 1,
        "gasto_medio": rng.normal(m, m * 0.2, k).clip(5),
        "perfil_real": nombre,
    }))
clientes = pd.concat(filas, ignore_index=True)
X = clientes[["recencia", "frecuencia", "gasto_medio"]]
clientes.describe().round(1)

## 2. ¿Por qué escalar?
k-means usa distancias euclídeas. Sin escalar, la recencia (cientos de días) domina sobre la frecuencia (decenas). `StandardScaler` deja cada variable con media 0 y desviación 1.

In [ ]:
X_esc = StandardScaler().fit_transform(X)

for nombre, datos in [("sin escalar", X.values), ("escalado", X_esc)]:
    etiquetas = KMeans(4, n_init=10, random_state=0).fit_predict(datos)
    print(f"{nombre:12s} → acuerdo con los perfiles reales (ARI): {adjusted_rand_score(clientes['perfil_real'], etiquetas):.2f}")

El *Adjusted Rand Index* (ARI) vale 1 si los grupos coinciden perfectamente con los reales y ~0 si son aleatorios. En la práctica **no tendrás** las etiquetas reales: por eso usamos criterios internos para elegir *k*.

## 3. Elegir el número de clusters

In [ ]:
ks = range(2, 10)
inercias, siluetas = [], []
for k in ks:
    km = KMeans(k, n_init=10, random_state=0).fit(X_esc)
    inercias.append(km.inertia_)
    siluetas.append(silhouette_score(X_esc, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].plot(ks, inercias, "o-"); axes[0].set(title="Método del codo", xlabel="k", ylabel="inercia")
axes[1].plot(ks, siluetas, "o-"); axes[1].set(title="Coeficiente de silueta", xlabel="k")
plt.tight_layout(); plt.show()

## 4. Interpretar los segmentos
Un segmento solo es útil si se puede **describir y accionar**.

In [ ]:
km = KMeans(4, n_init=10, random_state=0).fit(X_esc)
clientes["segmento"] = km.labels_
resumen = clientes.groupby("segmento")[["recencia", "frecuencia", "gasto_medio"]].mean().round(1)
resumen["clientes"] = clientes["segmento"].value_counts().sort_index()
resumen

In [ ]:
pd.crosstab(clientes["segmento"], clientes["perfil_real"])

In [ ]:
sns.pairplot(clientes, vars=["recencia", "frecuencia", "gasto_medio"], hue="segmento", palette="tab10", plot_kws={"s": 12, "alpha": 0.6})
plt.show()

## 5. Clustering jerárquico
No requiere fijar *k* de antemano: el dendrograma muestra a qué "altura" se fusionan los grupos.

In [ ]:
muestra = rng.choice(len(X_esc), 200, replace=False)
Z = linkage(X_esc[muestra], method="ward")
plt.figure(figsize=(12, 4))
dendrogram(Z, no_labels=True, color_threshold=12)
plt.title("Dendrograma (Ward, 200 clientes)"); plt.ylabel("distancia"); plt.show()

jer = AgglomerativeClustering(n_clusters=4, linkage="ward").fit_predict(X_esc)
print("ARI jerárquico vs perfiles reales:", round(adjusted_rand_score(clientes["perfil_real"], jer), 2))

## 6. Reducción de dimensión: dígitos manuscritos
Cada imagen tiene 8×8 = 64 píxeles → 64 dimensiones. ¿Podemos verlas en un plano?

In [ ]:
digits = load_digits()
fig, axes = plt.subplots(1, 10, figsize=(12, 1.5))
for ax, img, y in zip(axes, digits.images, digits.target):
    ax.imshow(img, cmap="gray_r"); ax.set_title(y); ax.axis("off")
plt.show()

### PCA
Busca las direcciones de **máxima varianza**. Es lineal, rápida e interpretable.

In [ ]:
pca = PCA().fit(digits.data)
var_acum = np.cumsum(pca.explained_variance_ratio_)
print("Componentes para explicar el 90 % de la varianza:", np.argmax(var_acum >= 0.9) + 1)

Z_pca = PCA(2).fit_transform(digits.data)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(var_acum); axes[0].axhline(0.9, ls="--", c="gray")
axes[0].set(title="Varianza explicada acumulada", xlabel="nº componentes")
sc = axes[1].scatter(*Z_pca.T, c=digits.target, cmap="tab10", s=8)
axes[1].legend(*sc.legend_elements(), title="dígito", bbox_to_anchor=(1, 1))
axes[1].set_title("PCA en 2D")
plt.tight_layout(); plt.show()

### t-SNE
No lineal: preserva **vecindades locales**. Muy útil para visualizar, pero las distancias entre grupos lejanos y los tamaños de los grupos **no** tienen significado.

In [ ]:
Z_tsne = TSNE(2, perplexity=30, random_state=0).fit_transform(digits.data)
plt.figure(figsize=(7, 6))
sc = plt.scatter(*Z_tsne.T, c=digits.target, cmap="tab10", s=8)
plt.legend(*sc.legend_elements(), title="dígito", bbox_to_anchor=(1, 1))
plt.title("t-SNE en 2D"); plt.show()

## Ejercicios

1. Pon un nombre de negocio a cada segmento de la sección 4 y propone **una acción de marketing** para cada uno.
2. Prueba `KMeans` con `k = 3` y `k = 6`. ¿Qué segmentos se fusionan o se dividen? ¿Cuál preferirías presentar a Dirección y por qué?
3. Aplica k-means con `k = 10` sobre los 64 píxeles de los dígitos y mide el ARI frente a `digits.target`. Repite sobre las 2 coordenadas de t-SNE. ¿Qué ocurre y por qué es peligroso clusterizar sobre t-SNE?
4. Añade una variable "antigüedad del cliente" y comprueba cómo cambian los segmentos.

In [ ]:
# TU CÓDIGO AQUÍ